# Agents That Don't Break
## Session 1: From LLM to Agent
**DMLS Agentic AI Workshop** · 120 minutes · *What companies actually hire for, built from scratch.*

---

### The story

It's your first Monday as the AI engineer at **Harbourline**, a small Halifax software company.
The support inbox has 4,000 unread tickets. Your manager drops one sentence on your desk:

> *"Make the inbox sort itself. And when it gets one wrong, I want to know **why**."*

By the end of today you'll have built a support-triage agent the way a production team would:
a model that returns structured answers your code can trust, retries and a fallback for when it fails,
two independent checks merged into one state, a trace for every decision, a LangGraph version,
and tests for the parts that need testing.

### How this notebook works

| Label | Meaning |
|---|---|
| **GIVEN** | Run it, read it, move on. |
| **EXAMPLE** | Same technique on a different problem (a pizza shop). Study it, then copy the *pattern*, not the code. |
| **YOUR TURN** | Replace every `raise NotImplementedError` with your own code. |
| **CHECK** | Run it after a YOUR TURN cell. If it prints PASS, move on. |

Run cells top to bottom. If something breaks, read the error and the trace before you read your code.

---
# Chapter 0 · Setup (do this before the session)

Three steps:

1. Get a **free** Groq API key at **[console.groq.com/keys](https://console.groq.com/keys)**. No credit card. **Use your own account**: free limits are per account, so a shared key runs out mid-lab.
2. In Colab, click the **Secrets** icon in the left sidebar, add a secret called `GROQ_API_KEY`, paste your key, and turn on notebook access.
3. Run the next three cells. The last one should print an answer in a few seconds.

| Provider | `MODEL =` | Secret name | Free limits (per account) |
|---|---|---|---|
| **Groq** (default) | `"groq/qwen/qwen3.8-27b"` | `GROQ_API_KEY` | 30 req/min · 1,000 req/day · 8K tokens/min · 200K tokens/day |
| **Mistral** (backup) | `"mistral/mistral-small-latest"` | `MISTRAL_API_KEY` | 1 req/sec · 500K tokens/min · 1B tokens/month |
| **OpenCode Zen** (experimental) | `"zen/big-pickle"` | `OPENCODE_API_KEY` | not published; "limited time" models |

> **Note:** Free tiers may use your prompts for training. Everything in this notebook uses made-up tickets, so that's fine. Never paste real customer data into a free tier.

**No key or no internet?** Set `MODEL = "mock"` below. You get a fake, offline "model" that answers using keyword rules. It's good enough to practise every exercise, but it's **not** an LLM, so the results in Chapters 1–3 won't mean much.

In [ ]:
# GIVEN: Install everything (about 30 seconds in Colab)
%pip install -q instructor openai langgraph pydantic ipytest

In [ ]:
# GIVEN: Pick your model. Change ONE line to switch provider; nothing else in the notebook changes.
MODEL = "groq/qwen/qwen3.8-27b"           # default: Groq free tier, no card
# MODEL = "mistral/mistral-small-latest"   # backup: Mistral free tier, biggest limits
# MODEL = "zen/big-pickle"                 # experimental: OpenCode Zen free model (limits unpublished)
# MODEL = "google/gemini-2.5-flash"        # optional: Google AI Studio (also run: %pip install "instructor[google-genai]")
# MODEL = "ollama/qwen2.5:7b"              # offline backup: Ollama on your laptop (Appendix A)
# MODEL = "mock"                           # no key / no internet: fake keyword model for practice

# Turn OFF hidden "thinking" for classification. Reasoning tokens are slow and eat your token limit.
# Qwen on Groq accepts "none"; if you switch Groq to an openai/gpt-oss model, use {"reasoning_effort": "low"}.
MODEL_KWARGS = {"reasoning_effort": "none"} if MODEL.startswith("groq/qwen/") else {}

import os, getpass

def _load_key(name):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
    except Exception:
        os.environ[name] = getpass.getpass(f"Paste your {name}: ")

KEY_NAMES = {"groq": "GROQ_API_KEY", "mistral": "MISTRAL_API_KEY", "zen": "OPENCODE_API_KEY", "google": "GOOGLE_API_KEY"}
_provider = MODEL.split("/", 1)[0]
if _provider in KEY_NAMES:
    _load_key(KEY_NAMES[_provider])
print("Using:", MODEL)

In [ ]:
# GIVEN: Helpers. You don't need to read this cell today, but you're welcome to.
import json, re, time, random, contextlib
from typing import Literal, Optional, get_args, get_origin

# ---------- 1. A fake offline model (only used when MODEL == "mock") ----------
_HINTS = {
    "urgent":    ["urgent", "asap", "immediately", "outage", " down"],
    "billing":   ["invoice", "charge", "charged", "refund", "billing", "payment", "bill"],
    "technical": ["crash", "error", "bug", "server", "login", "broken", "update", "down"],
    "positive":  ["love", "great", "amazing", "best"],
    "negative":  ["boring", "terrible", "awful", "worst", "hate"],
    "large":     ["large", "big", "family"],
    "small":     ["small", "personal"],
}

def _mock_value(name, ann, text):
    if get_origin(ann) is Optional or (get_origin(ann) is not None and type(None) in get_args(ann)):
        ann = [a for a in get_args(ann) if a is not type(None)][0]
    if get_origin(ann) is Literal:
        options = sorted(get_args(ann), key=lambda o: o != "urgent")  # most specific first
        for opt in options:                      # keyword hints first
            if any(w in text for w in _HINTS.get(opt, [])):
                return opt
        for opt in options:                      # then literal mentions
            if str(opt).lower() in text:
                return opt
        return get_args(ann)[-1]
    if ann is float: return 0.81
    if ann is int:   return 2
    if ann is bool:  return False
    if get_origin(ann) is list: return []
    return f"mock: decided from keywords in '{text[:40]}...'"

class _MockCompletions:
    def create(self, response_model=None, messages=None, **kwargs):
        text = (messages[-1]["content"] if messages else "").lower()
        if response_model is None:
            return "(mock model) I'm a keyword stub, not a real LLM."
        data = {n: _mock_value(n, f.annotation, text) for n, f in response_model.model_fields.items()}
        return response_model(**data)

class _MockClient:
    def __init__(self):
        self.chat = type("Chat", (), {"completions": _MockCompletions()})()

# ---------- 2. The structured-output client ----------
OPENAI_COMPATIBLE = {   # provider -> (base_url, key env var)
    "groq":    ("https://api.groq.com/openai/v1", "GROQ_API_KEY"),
    "mistral": ("https://api.mistral.ai/v1", "MISTRAL_API_KEY"),
    "zen":     ("https://opencode.ai/zen/v1", "OPENCODE_API_KEY"),
    "ollama":  ("http://localhost:11434/v1", None),
}

class _WithDefaults:
    """Adds MODEL_KWARGS (e.g. reasoning off) to every call, so your code never has to."""
    def __init__(self, inner, defaults):
        self._inner, self._defaults = inner, defaults
        outer = self
        class _Completions:
            def create(self, **kw):
                return outer._inner.chat.completions.create(**{**outer._defaults, **kw})
            def create_with_completion(self, **kw):
                return outer._inner.chat.completions.create_with_completion(**{**outer._defaults, **kw})
        self.chat = type("Chat", (), {"completions": _Completions()})()

def get_client():
    """Returns an Instructor client: you pass a Pydantic class, you get a validated object back."""
    if MODEL == "mock":
        return _MockClient()
    import instructor
    provider, name = MODEL.split("/", 1)
    if provider == "google":
        inner = instructor.from_provider(MODEL)
    else:
        # Groq, Mistral, Zen and Ollama all speak the OpenAI API: one code path for all of them.
        from openai import OpenAI
        base, key = OPENAI_COMPATIBLE[provider]
        mode = {"groq": instructor.Mode.TOOLS, "mistral": instructor.Mode.TOOLS,
                "zen": instructor.Mode.MD_JSON,   # free models: tool support unknown, ask for JSON instead
                "ollama": instructor.Mode.JSON}[provider]
        raw = OpenAI(base_url=base, api_key=os.environ[key] if key else "ollama")
        inner = _BindModel(instructor.from_openai(raw, mode=mode), name)
    return _WithDefaults(inner, MODEL_KWARGS)

class _BindModel:
    """from_openai clients need model= on every call; this fills it in."""
    def __init__(self, inner, name):
        self._inner, self._name = inner, name
        outer = self
        class _Completions:
            def create(self, **kw):
                return outer._inner.chat.completions.create(model=outer._name, **kw)
            def create_with_completion(self, **kw):
                return outer._inner.chat.completions.create_with_completion(model=outer._name, **kw)
        self.chat = type("Chat", (), {"completions": _Completions()})()

# ---------- 3. A plain, unstructured chat call (used in Chapters 1-3 only) ----------
def plain_chat(prompt, temperature=None):
    """Sends text, gets text back. No schema, no validation: the 'before' picture."""
    if MODEL == "mock":
        return "(mock model) Sure! I found a flight for $389 on SkyHigh Air, departing tomorrow 9am."
    provider, name = MODEL.split("/", 1)
    if provider == "google":
        from google import genai
        from google.genai import types
        cfg = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
        return genai.Client().models.generate_content(model=name, contents=prompt, config=cfg).text
    from openai import OpenAI
    base, key = OPENAI_COMPATIBLE[provider]
    c = OpenAI(base_url=base, api_key=os.environ[key] if key else "ollama")
    kw = {"temperature": temperature} if temperature is not None else {}
    if MODEL_KWARGS:
        kw["extra_body"] = dict(MODEL_KWARGS)   # non-standard params go in extra_body
    r = c.chat.completions.create(model=name, messages=[{"role": "user", "content": prompt}], **kw)
    return r.choices[0].message.content

def count_tokens(text):
    """Real token count from Gemini when available, otherwise a rough estimate (~4 chars per token)."""
    if MODEL.startswith("google/"):
        try:
            from google import genai
            return genai.Client().models.count_tokens(model=MODEL.split("/", 1)[1], contents=text).total_tokens
        except Exception:
            pass
    return max(1, round(len(text) / 4))

# ---------- 4. Chaos tools for Chapter 5 ----------
@contextlib.contextmanager
def simulate_outage(fn_name="classify_with_llm"):
    """Inside this block, the named function always fails, like the wifi just died."""
    g = globals()
    original = g[fn_name]
    def broken(*args, **kwargs):
        raise ConnectionError("simulated outage: the model API is unreachable")
    g[fn_name] = broken
    try:
        yield
    finally:
        g[fn_name] = original

# ---------- 5. Pretty printing ----------
def show(state):
    s = state if isinstance(state, dict) else state.model_dump()
    print(f"Ticket {s.get('ticket_id')}: {s.get('text')!r}")
    for k in ["category", "confidence", "validated", "compliance_flag", "priority"]:
        if k in s and s[k] is not None:
            print(f"   {k:16} {s[k]}")
    print(f"   {'response':16} {s.get('response')}")

def show_trace(state):
    trace = state["trace"] if isinstance(state, dict) else state.trace
    for i, line in enumerate(trace, 1):
        try:
            entry = json.loads(line)
            print(f"{i:>2}. [log] " + "  ".join(f"{k}={v}" for k, v in entry.items()))
        except (json.JSONDecodeError, TypeError):
            print(f"{i:>2}. {line}")

# ---------- 6. The inbox ----------
TICKETS = {
    "T-001": "My invoice #4521 charged me twice.",
    "T-002": "My server keeps crashing after the last update.",
    "T-003": "What are your office hours over the holidays?",
    "T-004": "hi",
    "T-005": "My server keeps crashing, this is urgent",
    "T-006": "I was billed twice and my lawyer says this is grounds for legal action.",
    "T-007": "Our production server is down, this is urgent",
}

client = get_client()
print("PASS: Helpers loaded.", len(TICKETS), "tickets in the inbox.")

In [ ]:
# GIVEN: Smoke test: if this prints an answer, you're ready for the session.
t0 = time.time()
print(plain_chat("In one short sentence: what is a support ticket?"))
print(f"Time: {time.time() - t0:.1f}s  (over ~5s every time? tell the instructor)")

---
# Chapter 1 · What an LLM is actually doing
*Time: 0:00 – 0:15 · Slides 1–4*

Before you build anything around a model, you need five facts about it. Each one becomes a design decision later today.

1. A **token** is a chunk of text, roughly a word or word-piece.
2. The model **predicts one token at a time**, based on every token before it.
3. **Temperature** controls how random that prediction is.
4. The **context window** is everything the model can see right now. Nothing else exists to it.
5. **Nothing persists between calls** unless your code carries it forward.

Run the next three cells and watch each fact happen.

In [ ]:
# GIVEN: Fact 1: tokens. Tokens, not words, are what you pay for and what fills the context window.
for s in ["Hello", "Halifax", "unbelievably", "My invoice #4521 charged me twice."]:
    print(f"{count_tokens(s):>3} tokens  ←  {s!r}")

In [ ]:
# GIVEN: Facts 2 + 3: one token at a time, with randomness you control.
prompt = "Invent a name for a lighthouse café in Nova Scotia. Reply with the name only."
for temp in [0.0, 1.5]:
    names = [plain_chat(prompt, temperature=temp).strip() for _ in range(3)]
    print(f"temperature={temp}: {names}")
# Low temperature: (nearly) the same answer every time. High temperature: variety.
# For routing tickets, which do you want?

In [ ]:
# GIVEN: Facts 4 + 5: the model has no memory. Two separate calls know nothing about each other.
print("Call 1:", plain_chat("My name is Akash. Just say 'nice to meet you'."))
print("Call 2:", plain_chat("What is my name?"))

# The ONLY way to 'remember' is for YOUR code to put the history back into the context:
history = "User: My name is Akash.\nAssistant: Nice to meet you.\nUser: What is my name?"
print("Call 3 (history carried by code):", plain_chat(history))

### The ceiling

Your manager's first test: *"Can't we just ask it to do things?"*
Here's the prompt, with no tools, no memory, and no way to check anything:

> **"Book me the cheapest flight under $400."**

Predict what happens, then run it.

In [ ]:
# GIVEN: The ceiling, live.
print(plain_chat("Book me the cheapest flight under $400."))

It either **invented** a flight or **admitted** it can't book one. Look back at the five facts:

- No ability to **act**: it can only produce text.
- No **memory**: it doesn't know who you are or where you live.
- No **goals** or way to check results: it predicts plausible tokens, then stops.

A plain LLM call is a text predictor. Everything after this point is **software engineering around that unreliable component**, and that's the skill job postings ask for: agent frameworks, structured output, evaluation and observability, failure handling.

---
# Chapter 2 · What are we actually building?
*Slides 5–11*

### Scope check

| System | What it does | Today? |
|---|---|---|
| Chatbot | Responds to text. No logic, no branching. | No |
| Pipeline | Fixed steps A → B → C. Reliable but rigid. | No |
| Workflow automation (Zapier) | Fixed steps triggered by events; branches only on predefined rules. | No |
| **Agent** | An LLM given *agency*: it produces a **decision for the system**, not a final answer for you. | Yes, the reliable kind |

### The agent loop

```
        ┌──────────────┐
   ┌───> │   OBSERVE    │  what's the state? what just happened?
   │    └──────┬───────┘
   │           ▼
   │    ┌──────────────┐
   │    │    THINK     │  decide what to do next   ← the LLM lives here
   │    └──────┬───────┘
   │           ▼
   │    ┌──────────────┐
   └─── │     ACT      │  do it, then observe the result
        └──────────────┘
```

### Core components

| Component | Job | Today |
|---|---|---|
| **Model** | The reasoning engine | Gemini, via a structured call |
| **Tools** | The hands | Not built today |
| **Memory** | What it remembers | The **state** object |
| **Controller** | The code deciding what happens next | **What you're building** |

### The split nobody tells you about

Most production systems are **not** autonomous agents looping freely. They're **state machines with an LLM plugged into specific nodes**, because a free-running loop is unpredictable and expensive to debug. (A coding agent like OpenCode is the free-running kind. A support-triage system is almost always the state-machine kind.)

Choosing the boring, auditable option over the flashy one, and being able to argue *why*, is a real design decision and a real interview answer.

### Routing: the LLM is the router, the code is the rails

We never let the LLM *execute* anything. We ask it to **classify** ("billing or technical?"), it returns a **structured decision**, and plain deterministic code (`if`/`else`, a dictionary) sends the ticket to the right function.

### Today's blueprint

```
ticket ──> [intake: LLM classifies] ──> [validate] ──> [compliance + priority checks] ──> [route] ──> handler
              │ structured output        rule            two rules, merged              rails
              │ + retries + fallback
              └──────────────── every step writes to the trace ────────────────┘
```

1. Force the LLM to output structured data our router can trust.
2. Build the state machine nodes (intake, validate, route).
3. Add failure handling, because LLMs fail.
4. Add observability, because debugging is a requirement.
5. Rebuild it in LangGraph, then test it.

---
# Chapter 3 · Structured output: making the model's answer trustworthy
*Time: 0:15 – 0:25*

First attempt at sorting the inbox: ask for a category in plain text, then search the reply for a keyword.
Run it and look at what comes back.

In [ ]:
# GIVEN: The fragile way: free text + string matching.
reply = plain_chat(f"Classify this support ticket as billing, technical, or general: {TICKETS['T-002']}")
print("Raw reply:\n", reply)

category = "billing" if "billing" in reply.lower() else "technical" if "technical" in reply.lower() else "general"
print("\nString matching decided:", category)
# What if the reply says "This is not a billing issue, it's technical"? Your parser says: billing.

That parser breaks the moment the model gets chatty. The fix is to **stop asking for text** and ask for **an object with a shape**.

You describe the shape with a **Pydantic** class. **Instructor** sends that shape to the model, **validates** the reply against it, and **automatically re-asks** the model if the reply doesn't fit.

> `response_model` isn't a suggestion to the model. It's a constraint your code enforces. That's the difference between *hoping* you got valid JSON and *knowing* you did.

`Literal[...]` means "only these exact values". `Field(ge=0, le=1)` means "a number between 0 and 1". If the model returns anything else, validation fails and Instructor retries.

In [ ]:
# EXAMPLE: a pizza shop turning messy order texts into objects.
from pydantic import BaseModel, Field

class PizzaOrder(BaseModel):
    size: Literal["small", "medium", "large"]   # only these three values are allowed
    toppings: list[str]
    confidence: float = Field(ge=0, le=1)       # validated: must be between 0 and 1
    notes: str

def read_order(text: str) -> PizzaOrder:
    return client.chat.completions.create(
        response_model=PizzaOrder,              # ← the constraint
        max_retries=2,                          # ← if validation fails, re-ask up to 2 times
        messages=[{"role": "user", "content": f"Extract this pizza order: {text}"}],
    )

order = read_order("yo can i get a big one w/ mushrooms and extra cheese, no onions pls")
print(order)
print("Type:", type(order).__name__, "| size field:", order.size)   # a real object, not a string

### YOUR TURN Your turn 3.1: the ticket schema

Write a Pydantic class called **`TicketClassification`** with three fields:

| Field | Type | Rule |
|---|---|---|
| `category` | one of `"billing"`, `"technical"`, `"general"` | use `Literal` |
| `confidence` | `float` | between 0 and 1 (use `Field`) |
| `reasoning` | `str` | the model's one-line explanation |

In [ ]:
# YOUR TURN 3.1: define TicketClassification (look at PizzaOrder above)
from pydantic import BaseModel, Field

class TicketClassification(BaseModel):
    # replace this line with your three fields
    pass

### YOUR TURN Your turn 3.2: the classifier

Write **`classify_with_llm(text) -> TicketClassification`**. It should look almost exactly like `read_order`, with your schema and a ticket-classification prompt.

In [ ]:
# YOUR TURN 3.2: write classify_with_llm (look at read_order above)
def classify_with_llm(text: str) -> TicketClassification:
    raise NotImplementedError("Exercise 3.2: call client.chat.completions.create with response_model=TicketClassification")

In [ ]:
# CHECK 3
assert "category" in TicketClassification.model_fields, "TicketClassification needs a 'category' field"
result = classify_with_llm(TICKETS["T-002"])
assert isinstance(result, TicketClassification), "classify_with_llm should return a TicketClassification"
assert result.category in ("billing", "technical", "general")
assert 0 <= result.confidence <= 1
print("PASS: Chapter 3 done:", result)

---
# Chapter 4 · The state machine, node by node
*Time: 0:25 – 0:40 · Notebook Part 1*

Now you have a trustworthy classifier. Next you wrap it in a **state machine**:

- **State**: one typed object holding everything known about a ticket. It's the agent's memory.
- **Nodes**: plain functions with the shape **state in → state out**. Each node does one job and writes a line to the trace.
- **Controller**: the code that calls the nodes in order.

The most important design decision in this chapter: **which nodes get the LLM, and which stay deterministic?** Not everything needs a model. Every model call costs money and latency and can fail.

Study the pizza version first. Every piece you're about to write has a twin here.

In [ ]:
# EXAMPLE: the pizza shop as a state machine.

# STATE: one typed object, carried through every node.
class OrderState(BaseModel):
    order_id: str
    text: str
    size: Optional[Literal["small", "medium", "large", "invalid"]] = None
    confidence: Optional[float] = None
    address_ok: bool = False
    message: Optional[str] = None
    trace: list[str] = []

# NODE 1 (uses the LLM): state in, state out, plus a trace line.
def take_order(state: OrderState) -> OrderState:
    o = read_order(state.text)
    state.size = o.size
    state.confidence = o.confidence
    state.trace.append(f"take_order: size '{o.size}' (confidence={o.confidence:.2f}): {o.notes}")
    return state

# NODE 2 (deterministic on purpose: a rule doesn't need a model).
def check_address(state: OrderState) -> OrderState:
    state.address_ok = "street" in state.text.lower() or "st." in state.text.lower()
    state.trace.append(f"check_address: {'ok' if state.address_ok else 'missing'}")
    return state

# NODE 3 (routing): a gate first, then a dictionary of handlers. The code is the rails.
def bake_small(state):  state.message = "Small oven, 10 min.";      state.trace.append("dispatch: small oven"); return state
def bake_medium(state): state.message = "Main oven, 14 min.";       state.trace.append("dispatch: main oven");  return state
def bake_large(state):  state.message = "Deck oven, 18 min.";       state.trace.append("dispatch: deck oven");  return state

def dispatch(state: OrderState) -> OrderState:
    if not state.address_ok:
        state.message = "Order held: no delivery address."
        state.trace.append("dispatch: held at gate")
        return state
    handlers = {"small": bake_small, "medium": bake_medium, "large": bake_large}
    return handlers[state.size](state)

# CONTROLLER: call the nodes in order.
def run_pizza_bot(order_id: str, text: str) -> OrderState:
    state = OrderState(order_id=order_id, text=text)
    state = take_order(state)      # observe + think (LLM)
    state = check_address(state)   # think (rule)
    state = dispatch(state)        # act
    return state

p = run_pizza_bot("P-1", "large pepperoni to 12 Spring Garden Street please")
print(p.message)
for line in p.trace: print("  ", line)

### YOUR TURN Your turn 4.1: the ticket state

Write **`TicketState`**, the twin of `OrderState`:

| Field | Type | Default |
|---|---|---|
| `ticket_id` | `str` | (required) |
| `text` | `str` | (required) |
| `category` | `Optional[Literal["billing", "technical", "general", "invalid"]]` | `None` |
| `confidence` | `Optional[float]` | `None` |
| `validated` | `bool` | `False` |
| `response` | `Optional[str]` | `None` |
| `trace` | `list[str]` | `[]` |

In [ ]:
# YOUR TURN 4.1: define TicketState
class TicketState(BaseModel):
    pass  # replace with the seven fields from the table

### YOUR TURN Your turn 4.2 + 4.3: the first two nodes

- **`intake(state)`** calls `classify_with_llm`, copies `category` and `confidence` into the state, and appends a trace line that includes the model's reasoning, e.g. `intake: classified as 'billing' (confidence=0.92): mentions invoice`.
- **`validate(state)`** is **deterministic, no LLM**. If the stripped text is shorter than 5 characters, set `category` to `"invalid"`, `validated` to `False`, and trace `validate: rejected, ticket text too short`. Otherwise set `validated = True` and trace `validate: passed`.

In [ ]:
# YOUR TURN 4.2: intake (twin: take_order)
def intake(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 4.2")

# YOUR TURN 4.3: validate (twin: check_address). No LLM here!
def validate(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 4.3")

The three handlers are given. They're boring on purpose: in a real company these would open a ticket in Zendesk, page someone, and so on.

In [ ]:
# GIVEN: The three handlers
def handle_billing(state: TicketState) -> TicketState:
    state.response = "Routed to Billing team. A refund review will follow within 2 business days."
    state.trace.append("respond: handled by billing")
    return state

def handle_technical(state: TicketState) -> TicketState:
    state.response = "Routed to Technical team. Logs are being pulled for your account."
    state.trace.append("respond: handled by technical")
    return state

def handle_general(state: TicketState) -> TicketState:
    state.response = "Routed to General support. Expect a reply within 24 hours."
    state.trace.append("respond: handled by general")
    return state

### YOUR TURN Your turn 4.4 + 4.5: route and the controller

- **`route(state)`**: if the ticket isn't validated, set `response = "Ticket rejected: not enough information."`, trace `route: rejected at gate`, and return. Otherwise look up the handler in a dictionary keyed by category and call it. (Twin: `dispatch`.)
- **`run_agent(ticket_id, text)`**: build a `TicketState`, run `intake` → `validate` → `route`, return the state. (Twin: `run_pizza_bot`.)

In [ ]:
# YOUR TURN 4.4: route
def route(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 4.4")

# YOUR TURN 4.5: the controller
def run_agent(ticket_id: str, text: str) -> TicketState:
    raise NotImplementedError("Exercise 4.5")

In [ ]:
# CHECK 4: run the whole inbox through your agent
for tid in ["T-001", "T-002", "T-003", "T-004"]:
    s = run_agent(tid, TICKETS[tid])
    show(s)
    print()
assert run_agent("T-004", TICKETS["T-004"]).validated is False, "'hi' should be rejected by validate"
print("PASS: Chapter 4 done. Now look at one trace:")
show_trace(run_agent("T-001", TICKETS["T-001"]))

> **Interview line:** *"I put the model only in the intake node. Validation and routing are deterministic, so they're cheap, fast, and testable. Knowing which nodes need the model and which don't is part of the job."*

---
# Chapter 5 · Error handling: what breaks in production
*Time: 0:40 – 0:50*

Tuesday morning. The model API has an outage. Here's what your agent does when the classifier can't reach the model:

In [ ]:
# GIVEN: Break it. (Instructor: this is also where you kill the wifi live.)
with simulate_outage():
    try:
        run_agent("T-001", TICKETS["T-001"])
    except Exception as e:
        print(f"CRASH: The whole agent crashed: {type(e).__name__}: {e}")

One flaky call took down the whole system. Production systems handle this with two tools:

1. **Retry with exponential backoff.** Try again, waiting 1s, then 2s, then 4s. Short outages and rate limits often clear on their own, and backing off stops you hammering a struggling API.
2. **A fallback path.** If every retry fails, **degrade gracefully** with a dumber method that always works, instead of crashing.

The pizza shop's oven-scheduling API is flaky too. Here's the pattern:

In [ ]:
# EXAMPLE: retry + backoff + fallback for a flaky oven API.
def book_oven_slot(size):
    if random.random() < 0.7:                       # fails 70% of the time
        raise TimeoutError("oven scheduler timed out")
    return f"slot booked for a {size} pizza"

def book_oven_with_retry(size, max_attempts=3, base_delay=0.5):
    for attempt in range(1, max_attempts + 1):
        try:
            return book_oven_slot(size)
        except Exception as e:
            wait = base_delay * 2 ** (attempt - 1)       # 0.5s, 1s, 2s: exponential backoff
            print(f"  attempt {attempt} failed ({e}), retrying in {wait}s...")
            time.sleep(wait)
    print(f"  all {max_attempts} attempts failed, falling back")
    return "no slot: chef will bake it next in line"     # the fallback ALWAYS works

print("→", book_oven_with_retry("large"))

### YOUR TURN Your turn 5.1: the fallback classifier

Write **`fallback_keyword_classify(text) -> TicketClassification`**. It uses no LLM: plain keyword rules.

- `"invoice"` or `"charge"` in the lowercased text → `billing`
- `"error"`, `"bug"` or `"crash"` → `technical`
- otherwise → `general`

Return a `TicketClassification` with `confidence=0.5` and `reasoning` starting with `"fallback"`, e.g. `"fallback: keyword match, LLM unavailable"`. The low confidence is honest: downstream code and humans can see this answer came from the backup path.

In [ ]:
# YOUR TURN 5.1: fallback_keyword_classify
def fallback_keyword_classify(text: str) -> TicketClassification:
    raise NotImplementedError("Exercise 5.1")

### YOUR TURN Your turn 5.2 + 5.3: retry, then wire it in

- **`classify_with_retry(text, max_attempts=3, base_delay=1.0)`** tries `classify_with_llm` up to `max_attempts` times with exponential backoff (`base_delay * 2 ** (attempt - 1)`), printing each failure. If all attempts fail, it returns `fallback_keyword_classify(text)`. (Twin: `book_oven_with_retry`.)
- Then **redefine `intake`** so it calls `classify_with_retry` instead of `classify_with_llm`. Nothing else in `intake` changes.

> Note the two layers: Instructor's `max_retries` re-asks when the model returns **badly shaped** output. Your loop retries when the **call itself fails** (network, rate limit, outage). Different failures, different fixes.

In [ ]:
# YOUR TURN 5.2: classify_with_retry
def classify_with_retry(text: str, max_attempts: int = 3, base_delay: float = 1.0) -> TicketClassification:
    raise NotImplementedError("Exercise 5.2")

# YOUR TURN 5.3: redefine intake to use the safe version
def intake(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 5.3")

In [ ]:
# CHECK 5: same outage as before. This time the agent should survive (takes ~7 seconds of backoff).
assert fallback_keyword_classify("My invoice is wrong").category == "billing"
with simulate_outage():
    s = run_agent("T-001", TICKETS["T-001"])
show(s)
assert s.response and "fallback" in s.trace[0], "intake should have fallen back to keywords"
print("\nPASS: Chapter 5 done. The agent degraded instead of crashing.")

> **Interview line:** *"The fallback isn't 'crash with a stack trace'. It's the old keyword logic, flagged with confidence 0.5 so it's visible downstream. A system that degrades gracefully is worth more than one that only works when everything's perfect."*
>
> **Production footnote:** `except Exception` also catches your own bugs (a typo becomes a silent fallback). Real systems catch only network/rate-limit errors. Good question to raise in a code review.

---
# Chapter 6 · Branching and the reducer
*Time: 0:50 – 1:05 · Notebook Part 2*

Legal has a new rule: *any ticket that mentions a lawsuit goes straight to them.* And ops wants outages flagged as high priority.

Neither check depends on the other, or on the model. Each one is a **branch** that writes its own field to the shared state, and routing then reads the **merged** result. That merge is the **reducer** pattern.

In [ ]:
# EXAMPLE: two independent checks on a pizza order, merged into one state.
class OrderStateV2(OrderState):
    allergy_flag: Optional[bool] = None             # new
    rush: Optional[Literal["normal", "rush"]] = None  # new

def check_allergy(state):
    state.allergy_flag = any(w in state.text.lower() for w in ["allergy", "allergic", "celiac"])
    state.trace.append(f"allergy_check: flag={state.allergy_flag}")
    return state

def check_rush(state):
    state.rush = "rush" if any(w in state.text.lower() for w in ["asap", "hurry", "quick"]) else "normal"
    state.trace.append(f"rush_check: {state.rush}")
    return state

def run_order_checks(state):
    state = check_allergy(state)
    state = check_rush(state)
    return state    # both branches wrote to ONE state object: that shared write IS the reducer

def dispatch_v2(state):
    if state.allergy_flag:                        # an override beats normal routing
        state.message = "Sent to the allergen-safe station."
        return state
    state = dispatch(state)
    if state.rush == "rush":
        state.message = "[RUSH] " + state.message
    return state

o = run_order_checks(OrderStateV2(order_id="P-2", text="small, hurry, I'm allergic to nuts", size="small", address_ok=True))
print(dispatch_v2(o).message, "|", o.trace)

### YOUR TURN Your turn 6.1: extend the state

Redefine **`TicketState`** with everything it had in 4.1, plus two new fields:
- `compliance_flag: Optional[bool] = None`
- `priority: Optional[Literal["normal", "high"]] = None`

In [ ]:
# YOUR TURN 6.1: TicketState, version 2
class TicketState(BaseModel):
    pass  # all seven old fields + compliance_flag + priority

### YOUR TURN Your turn 6.2: the two branches and the merge

Both checks are **deterministic**: no LLM, no retry needed.

- **`check_compliance`**: `compliance_flag = True` if any of `["lawsuit", "lawyer", "legal action"]` appears in the lowercased text. Trace `compliance_check: flag=...`.
- **`check_priority`**: `priority = "high"` if any of `["immediately", "urgent", "down", "outage", "asap"]` appears, else `"normal"`. Trace `priority_check: priority=...`.
- **`run_branches`**: run both checks on the same state and return it. (Twin: `run_order_checks`.)

In [ ]:
# YOUR TURN 6.2: the branches and the reducer
def check_compliance(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 6.2")

def check_priority(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 6.2")

def run_branches(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 6.2")

### YOUR TURN Your turn 6.3: routing that reads the merged state

- **`route_v2`**: (1) not validated → rejected, same as before. (2) `compliance_flag` → `response = "Escalated to Legal/Compliance review."`, trace `route: compliance override`, return. (3) Otherwise call the normal handler, then prefix the response with `"[HIGH PRIORITY] "` if `priority == "high"`. (Twin: `dispatch_v2`.)
- **`run_agent_v2`**: `intake` → `validate` → `run_branches` → `route_v2`.

In [ ]:
# YOUR TURN 6.3: route_v2 and the new controller
def route_v2(state: TicketState) -> TicketState:
    raise NotImplementedError("Exercise 6.3")

def run_agent_v2(ticket_id: str, text: str) -> TicketState:
    raise NotImplementedError("Exercise 6.3")

In [ ]:
# CHECK 6
for tid in ["T-005", "T-006", "T-001"]:
    show(run_agent_v2(tid, TICKETS[tid])); print()
assert "Legal" in run_agent_v2("T-006", TICKETS["T-006"]).response, "lawyer ticket should go to Legal"
assert run_agent_v2("T-005", TICKETS["T-005"]).response.startswith("[HIGH PRIORITY]")
print("PASS: Chapter 6 done.")

---
# Chapter 7 · Observability: "why did it do that?"
*Time: still inside 0:50 – 1:05*

Wednesday. Your manager forwards a ticket that went to the wrong team: *"Why did it do that?"*

Your trace is a list of strings. That's a start, but you can't search or filter it across thousands of runs. You'll replace it with a **structured log per node**: which node ran, how long it took, and what it decided.

The tool for adding the same behaviour to many functions without editing each one is a **decorator**.

In [ ]:
# EXAMPLE: a decorator that times any pizza node, without touching the node's code.
import functools

def timed(fn):
    @functools.wraps(fn)            # keeps the original function's name
    def wrapper(state):
        start = time.time()
        result = fn(state)          # run the real node
        print(f"Time: {fn.__name__} took {(time.time() - start) * 1000:.1f} ms")
        return result
    return wrapper

timed_check = timed(check_address)
_ = timed_check(OrderState(order_id="P-3", text="medium to 5 Barrington Street"))

### YOUR TURN Your turn 7.1: `traced_node`

`timed` takes the function directly. Yours takes a **name first**, so it has one more layer: `traced_node("intake")(intake)`.

Write **`traced_node(node_name)`**. Its wrapper should run the node, measure `duration_ms` (rounded to 1 decimal), then append **one JSON line** to `result.trace`:

```python
{"node": node_name, "duration_ms": ..., "output_category": result.category, "confidence": result.confidence}
```

Use `json.dumps(...)` so every line has the same machine-readable shape.

In [ ]:
# YOUR TURN 7.1: traced_node (three layers: name → function → wrapper)
def traced_node(node_name):
    def decorator(fn):
        raise NotImplementedError("Exercise 7.1: return a wrapper(state) like timed() does")
    return decorator

In [ ]:
# GIVEN: Wrap the nodes. The guard stops double-wrapping if you re-run this cell.
if not hasattr(intake, "__wrapped__"):
    intake = traced_node("intake")(intake)
if not hasattr(validate, "__wrapped__"):
    validate = traced_node("validate")(validate)

# CHECK 7
result = run_agent_v2("T-005", TICKETS["T-005"])
show_trace(result)
assert any('"node": "intake"' in line for line in result.trace), "expected a JSON trace line from intake"
print("\nPASS: Chapter 7 done. You can now answer 'why did it do that?' by reading the trace.")

> **Interview line:** *"How would you debug an agent that made the wrong call in production?" Honest answer: "I'd look at the trace."* Tools like LangSmith and Langfuse do exactly this at scale (capture every node's input, output and timing, then give you a searchable UI). The concept is identical to what you just wrote.

---
# Chapter 8 · The LangGraph rebuild
*Time: 1:05 – 1:20 · Notebook Part 3*

Your controller is a function that calls nodes in order. That works for 5 nodes. At 50 nodes, the wiring itself becomes the bug.

**LangGraph** formalises the state machine you already built. Think of a whiteboard:

- **State**: your data schema (a `TypedDict` this time).
- **Nodes**: the circles. Just Python functions.
- **Edges**: the lines between them. `add_edge(a, b)` means "always go from a to b".
- **Conditional edges**: a line that branches. `add_conditional_edges(a, router_fn, {...})` calls `router_fn(state)`, which **returns the name of the next node**. This is "the LLM is the router, the code is the rails", drawn on the board.

One difference: LangGraph nodes **return a new dict** instead of mutating an object. `{**state, "key": value}` means "everything from state, with this key changed".

> We're not throwing away our engineering: structured output, retries and tracing all come with us.

In [ ]:
# EXAMPLE: the pizza shop as a LangGraph.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class PizzaGraphState(TypedDict):
    text: str
    size: str
    message: str
    trace: list

def g_take_order(state):
    o = read_order(state["text"])                                   # same structured call as before
    return {**state, "size": o.size, "trace": state["trace"] + [f"take_order: {o.size}"]}

def g_small(state): return {**state, "message": "Small oven.", "trace": state["trace"] + ["oven: small"]}
def g_big(state):   return {**state, "message": "Deck oven.",  "trace": state["trace"] + ["oven: deck"]}

def g_pick_oven(state) -> str:            # a router: returns the NAME of the next node
    return "small_oven" if state["size"] == "small" else "deck_oven"

pg = StateGraph(PizzaGraphState)
pg.add_node("take_order", g_take_order)
pg.add_node("small_oven", g_small)
pg.add_node("deck_oven", g_big)
pg.add_edge(START, "take_order")
pg.add_conditional_edges("take_order", g_pick_oven, {"small_oven": "small_oven", "deck_oven": "deck_oven"})
pg.add_edge("small_oven", END)
pg.add_edge("deck_oven", END)
pizza_app = pg.compile()

out = pizza_app.invoke({"text": "one small margherita", "trace": []})
print(out["message"], out["trace"])
print(pizza_app.get_graph().draw_mermaid())   # paste into mermaid.live to see the picture

In [ ]:
# GIVEN: The ticket graph's state and the nodes that are straight ports of your earlier code.
class GraphState(TypedDict):
    ticket_id: str
    text: str
    category: str
    confidence: float
    validated: bool
    compliance_flag: bool
    priority: str
    response: str
    trace: list

def lg_validate(state: GraphState) -> GraphState:
    ok = len(state["text"].strip()) >= 5
    return {**state, "validated": ok, "trace": state["trace"] + [f"validate: {'passed' if ok else 'rejected'}"]}

def lg_compliance(state: GraphState) -> GraphState:
    flag = any(w in state["text"].lower() for w in ["lawsuit", "lawyer", "legal action"])
    return {**state, "compliance_flag": flag, "trace": state["trace"] + [f"compliance: flag={flag}"]}

def lg_priority(state: GraphState) -> GraphState:
    urgent = ["immediately", "urgent", "down", "outage", "asap"]
    p = "high" if any(w in state["text"].lower() for w in urgent) else "normal"
    return {**state, "priority": p, "trace": state["trace"] + [f"priority: {p}"]}

def _reply(state, text, who):
    prefix = "[HIGH PRIORITY] " if state.get("priority") == "high" else ""
    return {**state, "response": prefix + text, "trace": state["trace"] + [f"respond: handled by {who}"]}

def lg_billing(state):   return _reply(state, "Routed to Billing.", "billing")
def lg_technical(state): return _reply(state, "Routed to Technical.", "technical")
def lg_general(state):   return _reply(state, "Routed to General.", "general")
def lg_rejected(state):  return {**state, "response": "Ticket rejected: not enough information.", "trace": state["trace"] + ["respond: rejected at gate"]}
def lg_legal(state):     return {**state, "response": "Escalated to Legal/Compliance review.", "trace": state["trace"] + ["respond: compliance override"]}

### YOUR TURN Your turn 8.1: the intake node and the router

- **`lg_intake(state)`**: call `classify_with_retry(state["text"])` (structured output **and** retries, unchanged). Return a new dict with `category`, `confidence`, and a trace line `intake: <category> (confidence=0.92)` added. (Twin: `g_take_order`.)
- **`lg_route(state) -> str`**: a **router**, not a node. Return the name of the next node: `"rejected"` if not validated, `"legal"` if `compliance_flag`, otherwise the category (`"billing"`, `"technical"` or `"general"`). (Twin: `g_pick_oven`.)

In [ ]:
# YOUR TURN 8.1: lg_intake and lg_route
def lg_intake(state: GraphState) -> GraphState:
    raise NotImplementedError("Exercise 8.1")

def lg_route(state: GraphState) -> str:
    raise NotImplementedError("Exercise 8.1")

### YOUR TURN Your turn 8.2: wire the graph

Write **`build_graph()`** that returns a **compiled** app:

1. `StateGraph(GraphState)`
2. `add_node` for: `intake`, `validate`, `compliance`, `priority`, and the five handlers `billing`, `technical`, `general`, `rejected`, `legal`
3. Edges: `START → intake → validate → compliance → priority`
4. From `priority`, a **conditional edge** using `lg_route`, mapping each name it can return to its node
5. Every handler → `END`
6. `return graph.compile()`

In [ ]:
# YOUR TURN 8.2: build_graph
HANDLER_NODES = {"billing": lg_billing, "technical": lg_technical, "general": lg_general,
                 "rejected": lg_rejected, "legal": lg_legal}

def build_graph():
    raise NotImplementedError("Exercise 8.2: follow the pizza graph above")

In [ ]:
# CHECK 8
app = build_graph()
for tid in ["T-007", "T-001", "T-006", "T-004"]:
    r = app.invoke({"ticket_id": tid, "text": TICKETS[tid], "trace": []})
    print(f"{tid}: {r['response']}")
    print("      trace:", r["trace"])
assert app.invoke({"ticket_id": "x", "text": TICKETS["T-007"], "trace": []})["response"].startswith("[HIGH PRIORITY]")
print("\nPASS: Chapter 8 done. Your graph:\n")
print(app.get_graph().draw_mermaid())

> **Interview line:** *"Every node is a function I already trusted: structured output, retry, tracing all carried over. LangGraph didn't replace my engineering; it gave me `add_node`, `add_edge` and `compile()` instead of wiring the calls by hand. That's the entire value of a framework."*
>
> **Aside:** our compliance and priority checks run one after another. True parallel fan-out is just `add_edge` from one node to **two** nodes that both feed a shared next node. Worth knowing; not today's lab.

---
# Chapter 9 · Testing something non-deterministic
*Time: 1:20 – 1:30*

You can't assert the exact output of an LLM call: the same input can get a slightly different answer twice. So you use **two testing philosophies in one file**:

| What you're testing | Philosophy | Example |
|---|---|---|
| Anything with an LLM in it | **Test the shape**: valid category, confidence in range, non-empty reasoning | `test_classification_returns_valid_category` |
| Anything deterministic | **Test exactly**, every time, no excuses | `test_fallback_triggers_correctly` |

`%%ipytest` runs real **pytest** on the functions in a cell. Two tests are written for you, one of each kind.

In [ ]:
# GIVEN: Turn on pytest inside the notebook
import ipytest
ipytest.autoconfig()

In [ ]:
%%ipytest -q
# GIVEN: Two example tests are given. 9.1: write the other two at the bottom.

def test_classification_returns_valid_category():
    # LLM involved: don't assert WHICH category on ambiguous text, assert the SHAPE
    result = classify_with_llm("My server keeps crashing and I was also charged twice")
    assert result.category in ("billing", "technical", "general")
    assert 0.0 <= result.confidence <= 1.0
    assert len(result.reasoning) > 0

def test_fallback_triggers_correctly():
    # No LLM involved: assert exactly
    result = fallback_keyword_classify("My invoice is wrong")
    assert result.category == "billing"
    assert result.reasoning.startswith("fallback")

# YOUR TURN 9.1a: test that route() is deterministic.
#   Build a TicketState(ticket_id="T-1", text="x", category="billing", validated=True),
#   call route(), and assert "Billing" is in the response.
def test_routing_is_deterministic():
    raise NotImplementedError("Exercise 9.1a")

# YOUR TURN 9.1b: test that validate() rejects a too-short ticket like "hi".
def test_short_ticket_is_rejected():
    raise NotImplementedError("Exercise 9.1b")

> **Interview line:** *"Four tests, two philosophies. Knowing what I can and can't assert exactly is what separates 'I wrote some tests' from being able to defend my test suite in a code review."*

---
# Chapter 10 · The build exercise: a portfolio piece, not a stub
*Time: 1:30 – 1:55 · 25 minutes, hands on keyboard*

Friday. Ops wants a fourth path: **urgent**. Outages shouldn't wait behind billing questions.

> **The brief:** Add a fourth path, **`urgent`**. It needs its own schema value, its own node, its own routing branch, its own retry-safe classification, its own trace entries, and at least one test. When you're done, you should be able to explain every line of it in an interview.

### Checklist
- [ ] **Schema:** `TicketClassification.category` can now be `"urgent"` (a new `Literal` value is enough; no new field needed).
- [ ] **Retry-safe:** urgent tickets go through `classify_with_retry`, **not** a shortcut. And ask yourself: does your **fallback** know about urgent?
- [ ] **Node:** `lg_handle_urgent`, same shape as the other handlers, with its own trace entry.
- [ ] **Wiring:** the node is added to the graph **and actually reachable**.
- [ ] **Test:** one test, either "assert valid shape" or "assert exact behaviour". Be ready to explain why you picked that kind.

**Success looks like:** `T-007` ("Our production server is down, this is urgent") comes out with an urgent response, and its trace shows exactly how it got there.

> **Hint 1:** an outage is *also* a technical problem. How does the model know which one you mean? (Look up `Field(description=...)`; Instructor sends it to the model.)
>
> **Hint 2: if the urgent ticket doesn't end up where you expect:** read the trace (and the error message) before you read your code. Which was the last node that ran? What did intake decide?

In [ ]:
# YOUR TURN 10.1: extend the schema (and think about your fallback)
class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "general"]   # ← change me
    confidence: float = Field(ge=0, le=1)
    reasoning: str

In [ ]:
# YOUR TURN 10.2: the urgent handler node (twin: lg_billing, built on _reply)
def lg_handle_urgent(state: GraphState) -> GraphState:
    raise NotImplementedError("Exercise 10.2")

In [ ]:
# YOUR TURN 10.3: wire it in. Start from build_graph() and make the urgent node reachable.
def build_graph_v2():
    raise NotImplementedError("Exercise 10.3")

app_v2 = build_graph_v2()
r = app_v2.invoke({"ticket_id": "T-007", "text": TICKETS["T-007"], "trace": []})
print(r["response"])
show_trace(r)

In [ ]:
%%ipytest -q
# YOUR TURN 10.4: at least one test for the urgent path. Which philosophy fits, and why?
def test_urgent_path():
    raise NotImplementedError("Exercise 10.4")

In [ ]:
# CHECK 10: the urgent path, end to end, including during an outage
r = build_graph_v2().invoke({"ticket_id": "T-007", "text": TICKETS["T-007"], "trace": []})
assert "urgent" in " ".join(r["trace"]), "the trace should show the urgent handler ran"

with simulate_outage():
    r_out = build_graph_v2().invoke({"ticket_id": "T-007", "text": TICKETS["T-007"], "trace": []})
print("During an outage:", r_out["response"])
print("PASS: Chapter 10 done." if "urgent" in " ".join(r_out["trace"])
      else "WARNING: Works normally, but during an outage the urgent ticket went elsewhere. Look at your fallback.")

---
# Chapter 11 · The sentence
*Time: 1:55 – 2:00*

Not *"I built a chatbot."* This:

> ### *"I designed a stateful agent with structured output, retry and fallback logic, and full tracing, then reimplemented it in LangGraph and tested the parts that needed testing."*

Every word of that sentence is a cell you wrote today. Write it down in the cell below, in your own words if you like, and put it on your résumé project line.

*Double-click and write your sentence here:*

>

---
**Next session:** Session 2.

---
# Appendix A · Running on a local LLM (offline backup)

Use this if the venue internet dies, or if you'd rather not send anything to a cloud API. It runs on **your laptop**, not in Colab.

1. Install Ollama from **[ollama.com/download](https://ollama.com/download)**.
2. In a terminal: `ollama pull qwen2.5:7b` (≈4–5 GB, so do this at home, not on venue wifi).
3. Run this notebook in local Jupyter (`pip install notebook`, then `jupyter notebook`).
4. In Chapter 0, set `MODEL = "ollama/qwen2.5:7b"`.

Expect it to be slower than the API. Small local models are also worse at structured output, so you may see Instructor retries in Chapter 3. That's a real, useful lesson: the schema is enforced either way, but a weaker model needs more re-asks to meet it.

# Appendix B · How much free quota does this session use?

Measured by running the whole notebook once: **37 model calls** (25 structured, 12 plain), roughly 15K tokens. With re-runs and debugging, budget **~110 calls and ~60K tokens per student** for the full two hours, peaking around 6 calls in a few seconds when a PASS: check cell runs.

| Provider | Fits one student's workshop? | Your own practice runs per day (one account) |
|---|---|---|
| Groq (qwen, reasoning off) | Yes, ~3× headroom on tokens | ~3–4 full workshops (200K tokens/day is the cap) |
| Mistral free | Yes, huge headroom | effectively unlimited |
| OpenCode Zen free | Unknown: limits aren't published | run `provider_check.py` to find out |

**Every student needs their own key.** These limits are per account. Thirty people on one key get 1/30th each.

A rate-limit error looks exactly like an outage to your retry loop, so a student who hits the limit will silently get **fallback** answers (confidence 0.5). If you see lots of `fallback` in traces, that's quota, not a bug.

# Appendix C · Why `MODEL` is one line

You just switched provider without touching a single node, test, or graph. That's not a convenience; it's a design decision. Providers deprecate models, change free tiers, and go down. Code that hardcodes a provider breaks when they do.